In [1]:
# Datasets compuertas Lógicas (lineales)
# AND
X_train = [[1, 1],
          [1, 0],
          [0, 1],
          [0, 0],
         ]
y_train = [1,0,0,0]

In [3]:
# OR
X_train = [[1, 1],
          [1, 0],
          [0, 1],
          [0, 0],
         ]
y_train = [1,1,1,0]

In [5]:
# NOT
X_train = [[1],
          [0],
         ]
y_train = [0, 1]

In [7]:
# Modelo de Perceptron simple
import sklearn.metrics as metric
import numpy as np

In [9]:
# Red de neurona única
from sklearn.linear_model import Perceptron

neurona = Perceptron(max_iter=100)
neurona.fit(X_train, y_train)

Perceptron(max_iter=100)

In [11]:
neurona.intercept_

array([1.])

In [13]:
neurona.coef_

array([[-2.]])

In [15]:
y_pred = neurona.predict(X_train)
print(y_pred)

[0 1]


In [17]:
precision = metric.accuracy_score(y_train, y_pred, normalize=True)
print(f"Precision = {precision}")

Precision = 1.0


In [19]:
# Dataset es no Lineal (XNOR)
X_train = [[1, 1],
          [1, 0],
          [0, 1],
          [0, 0],
         ]
y_train = [1,0,0,1]

In [31]:
# Arquitectura de red Neuronal con Perceptron Multicapa
from sklearn.neural_network import MLPClassifier

red = MLPClassifier(
    solver = "lbfgs",
    hidden_layer_sizes = (2,1),
    activation = "logistic",
    max_iter = 500)

red.fit(X_train, y_train)

MLPClassifier(activation='logistic', hidden_layer_sizes=(2, 1), max_iter=500,
              solver='lbfgs')

In [33]:
print([inter.shape for inter in red.intercepts_])

[(2,), (1,), (1,)]


In [35]:
print([coef.shape for coef in red.coefs_])

[(2, 2), (2, 1), (1, 1)]


In [37]:
y_pred = red.predict(X_train)
print(y_pred)

[1 0 0 1]


In [39]:
precision = metric.accuracy_score(np.array(y_train).flatten(), np.array(y_pred).flatten(), normalize = True)
print(f"Precision = {precision}")

Precision = 1.0


# Framework Pytorch

In [168]:
import torch
import torch.nn as nn
import torch.optim as optim

class MLP(nn.Module):
    # Arquitectura o la estructura de las capas y las neuronas
    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(n_inputs, 2),
            nn.Sigmoid(),
            
            nn.Linear(2, 1),
            nn.Sigmoid(),
            
            nn.Linear(1, n_outputs)
        )
    def forward(self, x):
        return self.network(x)
    

red_torch = MLP(2, 1)
    

In [170]:
# Tranformar los datos a tensores
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)

In [172]:
# Confifurar los hiperparámetros
criterio = nn.BCEWithLogitsLoss()

solver = optim.LBFGS(
    red_torch.parameters(),
    max_iter = 5000
)

# Entrenamiento
def closure():
    solver.zero_grad()
    output = red_torch(X_tensor)
    loss = criterio(output, y_tensor)
    loss.backward()
    return loss

solver.step(closure)

tensor(0.7026, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)

In [174]:
red_torch.eval()

with torch.no_grad():
    logits = red_torch(X_tensor)
    prob = torch.sigmoid(logits)
    pred = (prob >= 0.5).int()
    
print("Probabilidades:")
print(prob)

print("\nPredicciones:")
print(pred)

Probabilidades:
tensor([[1.0000e+00],
        [1.2311e-07],
        [1.2311e-07],
        [1.0000e+00]])

Predicciones:
tensor([[1],
        [0],
        [0],
        [1]], dtype=torch.int32)


In [176]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_train,
    pred.numpy().flatten()
)

print("Accuracy:", accuracy)

Accuracy: 1.0


# Framework Keras/Tensorflow

In [218]:
import numpy as np

X_train = np.array([
    [1, 1],
    [1, 0],
    [0, 1],
    [0, 0]
], dtype=np.float32)

y_train = np.array([
    [1],
    [0],
    [0],
    [1]
], dtype=np.float32)

In [236]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

n_inputs = 2

red_tf = keras.Sequential([
    layers.Input(shape=(n_inputs,)),
    layers.Dense(2, activation = 'sigmoid'),
    layers.Dense(1, activation = 'sigmoid'),
    layers.Dense(1, activation = 'sigmoid')
])

red_tf.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 1)              │             2 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11 (44.00 B)

 Trainable params: 11 (44.00 B)

 Non-trainable params: 0 (0.00 B)

In [238]:
# Configuracion de Hiperparámetros
red_tf.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

# Entrenamiento
history = red_tf.fit(
    X_train,
    y_train,
    epochs = 5000,
    verbose = 0
)

In [239]:
prob = red_tf.predict(X_train)

pred = (prob >= 0.5).astype(int)

print(prob)
print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[[0.8737148 ]
 [0.10718421]
 [0.10456054]
 [0.926009  ]]
[[1]
 [0]
 [0]
 [1]]


In [242]:
accuracy = accuracy_score(
    y_train,
    pred
)

print("Accuracy:", accuracy)

Accuracy: 1.0
